In [ ]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.2 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-7B-Instruct"

# Configure 4-bit quantization to fit the free T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load Tokenizer and Model
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model (this might take a couple of minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
print("Model loaded successfully!")

Loading tokenizer...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully!


#SINHALA LANGUAGE

In [ ]:
import json

DATA_PATH = "/content/sample_data/sri_lankan_simplified.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} items")

Loaded 203 items


In [ ]:
INSTRUCTION_PROMPT = (
    """ You are a Sinhala Expert for answering multiple-choice questions.
    You need to always evaluate these questions based on the Sri Lankan context and provide the most accurate answer.
    You should only respond with the correct option text, without any additional explanation or commentary.

Question: {question}
Option A: {option A}
Option B: {option B}
Option C: පිළිතුරු දෙකම නිවැරදියි.
Option D: පිළිතුරු දෙකම නිවැරදි නොවේ.
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'.
"""
)

def build_prompt(item):
    lines = [INSTRUCTION_PROMPT, "", item["scenario_question"], "", "Options:"]
    for letter, text in item["options"].items():
        lines.append(f"{letter}. {text}")
    return "\n".join(lines)

In [ ]:
import re
def generate_answer(prompt, valid_letters, max_new_tokens=20):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    input_len = inputs["input_ids"].shape[-1]
    raw = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()

    # First try a clean letter match (model followed instructions)
    match = re.search(r"\b([A-D])\b", raw.upper())
    if match and match.group(1) in valid_letters:
        predicted = match.group(1)
    # Fall back: model echoed the option content instead of the letter
    elif "දෙකම නිවැරදි නොවේ" in raw:
        predicted = "0"
    elif "දෙකම නිවැරදි" in raw:   # check this AFTER the "නොවේ" (negative) case
        predicted = "Both"
    else:
        predicted = None

    return predicted, raw

In [ ]:
OUTPUT_PATH = "/content/sri_lankan_results.json"

for i, item in enumerate(data):
    if item.get("model_answer"):  # already processed, skip (resume-safe)
        continue

    prompt = build_prompt(item)
    predicted, raw_output = generate_answer(prompt, list(item["options"].keys()))

    item["model_answer"] = predicted
    item["raw_model_output"] = raw_output

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"[{i+1}/{len(data)}] {item['id']} -> {predicted} (gold: {item['gold_answer']})")

print("All done.")

[1/203] SI_2004 -> A (gold: A)
[2/203] SI_2007 -> A (gold: B)
[3/203] SI_2009 -> B (gold: A)
[4/203] SI_2012 -> A (gold: B)
[5/203] SI_2015 -> A (gold: A)
[6/203] SI_2017 -> A (gold: A)
[7/203] SI_2018 -> A (gold: A)
[8/203] SI_2022 -> A (gold: A)
[9/203] SI_2026 -> A (gold: B)
[10/203] SI_2027 -> A (gold: A)
[11/203] SI_2028 -> A (gold: A)
[12/203] SI_2033 -> A (gold: B)
[13/203] SI_2036 -> A (gold: A)
[14/203] SI_2038 -> A (gold: A)
[15/203] SI_2039 -> A (gold: A)
[16/203] SI_2046 -> A (gold: A)
[17/203] SI_2049 -> A (gold: B)
[18/203] SI_2056 -> A (gold: A)
[19/203] SI_2092 -> A (gold: A)
[20/203] SI_2097 -> A (gold: A)
[21/203] SI_2100 -> B (gold: Both)
[22/203] SI_2105 -> A (gold: A)
[23/203] SI_2111 -> A (gold: A)
[24/203] SI_2112 -> A (gold: A)
[25/203] SI_2124 -> A (gold: A)
[26/203] SI_2127 -> A (gold: A)
[27/203] SI_2129 -> A (gold: A)
[28/203] SI_2132 -> A (gold: B)
[29/203] SI_2151 -> A (gold: A)
[30/203] SI_2157 -> A (gold: A)
[31/203] SI_2158 -> A (gold: A)
[32/203] SI_21

In [ ]:
import json

with open("/content/sri_lankan_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

correct = sum(1 for item in results if item.get("model_answer") == item.get("gold_answer"))
total = len(results)

print(f"Correct: {correct} / {total}")
print(f"Accuracy: {correct / total:.2%}")

Correct: 104 / 203
Accuracy: 51.23%


#CHINESE LANGUAGE

In [ ]:
import json

DATA_PATH = "/content/sample_data/chinese_simplified.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} items")

Loaded 790 items


In [ ]:
INSTRUCTION_PROMPT = (
    """
You are a Chinese Expert for answering multiple-choice questions.
You need to always evaluate these questions based on the Chinese context and provide the most accurate answer.
You should only respond with the correct option text, without any additional explanation or commentary.

Question: {question}
Option A: {option A}
Option B: {option B}
Option C: {option C}
Option D: {option D}
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'. "Question": "

"""
)

def build_prompt(item):
    lines = [INSTRUCTION_PROMPT, "", item["scenario_question"], "", "Options:"]
    for letter, text in item["options"].items():
        lines.append(f"{letter}. {text}")
    return "\n".join(lines)

In [ ]:
import re
def generate_answer(prompt, valid_letters, max_new_tokens=20):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    input_len = inputs["input_ids"].shape[-1]
    raw = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()

    # First try a clean letter match (model followed instructions)
    match = re.search(r"\b([A-D])\b", raw.upper())
    if match and match.group(1) in valid_letters:
        predicted = match.group(1)
    else:
        predicted = None

    return predicted, raw

In [ ]:
OUTPUT_PATH = "/content/chinese_results.json"

for i, item in enumerate(data):
    if item.get("model_answer"):  # already processed, skip (resume-safe)
        continue

    prompt = build_prompt(item)
    predicted, raw_output = generate_answer(prompt, list(item["options"].keys()))

    item["model_answer"] = predicted
    item["raw_model_output"] = raw_output

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"[{i+1}/{len(data)}] {item['id']} -> {predicted} (gold: {item['gold_answer']})")

print("All done.")

[1/790] ZH_0_0 -> B (gold: A)
[2/790] ZH_0_3 -> C (gold: C)
[3/790] ZH_0_41 -> A (gold: A)
[4/790] ZH_0_42 -> A (gold: B)
[5/790] ZH_0_64 -> A (gold: A)
[6/790] ZH_1_47 -> A (gold: A)
[7/790] ZH_1_57 -> A (gold: A)
[8/790] ZH_1_58 -> A (gold: A)
[9/790] ZH_1_69 -> B (gold: A)
[10/790] ZH_1_77 -> A (gold: A)
[11/790] ZH_2_27 -> A (gold: A)
[12/790] ZH_2_48 -> B (gold: A)
[13/790] ZH_2_57 -> C (gold: A)
[14/790] ZH_2_60 -> C (gold: B)
[15/790] ZH_2_69 -> B (gold: A)
[16/790] ZH_3_14 -> A (gold: C)
[17/790] ZH_3_15 -> A (gold: A)
[18/790] ZH_3_16 -> A (gold: A)
[19/790] ZH_3_22 -> C (gold: C)
[20/790] ZH_3_79 -> C (gold: B)
[21/790] ZH_4_9 -> B (gold: B)
[22/790] ZH_4_23 -> B (gold: C)
[23/790] ZH_4_38 -> B (gold: D)
[24/790] ZH_4_46 -> B (gold: B)
[25/790] ZH_4_74 -> A (gold: A)
[26/790] ZH_5_14 -> A (gold: A)
[27/790] ZH_5_22 -> A (gold: A)
[28/790] ZH_5_73 -> A (gold: A)
[29/790] ZH_5_75 -> D (gold: D)
[30/790] ZH_5_76 -> A (gold: A)
[31/790] ZH_6_20 -> A (gold: A)
[32/790] ZH_6_46 -> 

In [ ]:
import json

with open("/content/chinese_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

correct = sum(1 for item in results if item.get("model_answer") == item.get("gold_answer"))
total = len(results)

print(f"Correct: {correct} / {total}")
print(f"Accuracy: {correct / total:.2%}")

Correct: 320 / 790
Accuracy: 40.51%


#INDONESIAN LANGUAGE

In [ ]:
import json

DATA_PATH = "/content/sample_data/indonesian_simplified.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} items")

Loaded 366 items


In [ ]:
INSTRUCTION_PROMPT = (
    """
    You are an Indonesian Expert for answering multiple-choice questions.
    You need to always evaluate these questions based on the Indonesian context and provide the most accurate answer.
    You should only respond with the correct option text, without any additional explanation or commentary.

Question: {question}
Option A: {option A}
Option B: {option B}
Option C: {option C}
Option D: {option D}
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'.


"""
)

def build_prompt(item):
    lines = [INSTRUCTION_PROMPT, "", item["scenario_question"], "", "Options:"]
    for letter, text in item["options"].items():
        lines.append(f"{letter}. {text}")
    return "\n".join(lines)

In [ ]:
import re
def generate_answer(prompt, valid_letters, max_new_tokens=20):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    input_len = inputs["input_ids"].shape[-1]
    raw = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()

    # First try a clean letter match (model followed instructions)
    match = re.search(r"\b([A-D])\b", raw.upper())
    if match and match.group(1) in valid_letters:
        predicted = match.group(1)
    else:
        predicted = None

    return predicted, raw

In [ ]:
OUTPUT_PATH = "/content/indonesian_results.json"

for i, item in enumerate(data):
    if item.get("model_answer"):  # already processed, skip (resume-safe)
        continue

    prompt = build_prompt(item)
    predicted, raw_output = generate_answer(prompt, list(item["options"].keys()))

    item["model_answer"] = predicted
    item["raw_model_output"] = raw_output

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"[{i+1}/{len(data)}] {item['id']} -> {predicted} (gold: {item['gold_answer']})")

print("All done.")

[1/366] ID_3 -> A (gold: A)
[2/366] ID_5 -> B (gold: B)
[3/366] ID_6 -> D (gold: D)
[4/366] ID_9 -> A (gold: A)
[5/366] ID_15 -> A (gold: A)
[6/366] ID_17 -> A (gold: A)
[7/366] ID_24 -> D (gold: D)
[8/366] ID_28 -> D (gold: A, D)
[9/366] ID_36 -> B (gold: B)
[10/366] ID_38 -> B (gold: C)
[11/366] ID_42 -> C (gold: C)
[12/366] ID_47 -> A (gold: C, D)
[13/366] ID_52 -> B (gold: B)
[14/366] ID_58 -> A (gold: A)
[15/366] ID_61 -> A (gold: A)
[16/366] ID_74 -> A (gold: A)
[17/366] ID_91 -> D (gold: A)
[18/366] ID_97 -> A (gold: C)
[19/366] ID_98 -> A (gold: A)
[20/366] ID_102 -> C (gold: C)
[21/366] ID_108 -> B (gold: A, D)
[22/366] ID_110 -> A (gold: B)
[23/366] ID_115 -> C (gold: C)
[24/366] ID_117 -> B (gold: D)
[25/366] ID_133 -> B (gold: A)
[26/366] ID_137 -> C (gold: C)
[27/366] ID_146 -> D (gold: C)
[28/366] ID_148 -> B (gold: D)
[29/366] ID_149 -> A (gold: A)
[30/366] ID_152 -> D (gold: B, D)
[31/366] ID_155 -> A (gold: D)
[32/366] ID_156 -> B (gold: C, D)
[33/366] ID_158 -> C (gol

In [ ]:
import json

with open("/content/indonesian_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

correct = 0
for item in results:
    model_answer = item.get("model_answer")
    gold_answer = item.get("gold_answer", "")

    gold_letters = [g.strip() for g in gold_answer.split(",")]  # handles single or multiple gold answers

    if model_answer is not None and model_answer.strip() in gold_letters:
        correct += 1

total = len(results)

print(f"Correct: {correct} / {total}")
print(f"Accuracy: {correct / total:.2%}")

Correct: 235 / 366
Accuracy: 64.21%
